<a href="https://colab.research.google.com/github/Zhengdao-Lu/Midterm-Project-Econometric-Foundation/blob/main/midterm_starter_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 Midterm Project — Econometric Foundation

**ECON 3916 · Applied Data Analytics** · Topics 1-14 · 100 points · your role: Independent analyst

This is the **starter**: the structure of the project in rubric order. Fill the empty cells and keep every heading so the grader finds each section. The brief on Canvas has the full text of every section and the rubric. The grader reruns this notebook from the top, so every number, table and figure in your report must come from a cell here.

**🛑 This project is entirely NO AI.** Not phases one to three — all of it. No AI for the code, the analysis, the writing, or the editing. This is the one assessment in the course that establishes you can do the work unaided, and it is the one you will draw on when an interviewer asks what you built. Everything you have practised with a co-pilot, you do here alone.

## Start here

Work top to bottom. What each part of this notebook is for:

1. **Data provenance**: where the file came from. It feeds Section 1 and your README.
2. **Section 1**: load the data into a DataFrame called `df`, then run the *Section 1 self-check* cell and answer the questions under it. Do this before the **Mon 2 Nov** checkpoint.
3. **Section 2**: figures, each with a caption that states a finding.
4. **Section 3**: the OLS model, with a draft committed by **Fri 6 Nov**.
5. **Section 4**: diagnostics, only after Topic 14.
6. **Section 5**: words only.
7. **Numbers my report quotes**: last. Every number in the report must be printed there.

The brief's *Six checks before you post* list the problems that came up most often in Checkpoint 1. Read them before you settle on your controls.

## Setup

Imports only. Add any others you need here, at the top.

In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [50]:
import os
import urllib.request
from urllib.parse import quote

REPO_RAW = "https://raw.githubusercontent.com/Zhengdao-Lu/Midterm-Project-Econometric-Foundation/main/data/"

files = [
    "GDP per capita.csv",
    "Current health expenditure per capita.csv",
    "Life expectancy .csv",
    "Urban population.csv"
]

os.makedirs("data", exist_ok=True)

for name in files:
    path = "data/" + name
    if not os.path.exists(path):
        urllib.request.urlretrieve(REPO_RAW + quote(name), path)

print(sorted(os.listdir("data")))

['Current health expenditure per capita.csv', 'GDP per capita.csv', 'Life expectancy .csv', 'Urban population.csv']


## Data provenance

Fill these in when you download the data. They feed Section 1 and your README.

In [51]:
SOURCE_URL = {"GDP per capita":"https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.KD",
              "Current health expenditure per capita":"https://data.worldbank.org/indicator/SH.XPD.CHEX.PP.CD",
              "Life expectancy":"https://data.worldbank.org/indicator/SP.DYN.LE00.IN",
              "Urban population":"https://data.worldbank.org/indicator/SP.URB.TOTL.IN.ZS"
              }       # the page or file you downloaded from

DOWNLOAD_DATE = "2026-11-04"    # YYYY-MM-DD

FILE_NAME =  {
    "GDP per capita": "data/GDP per capita.csv",
    "Current health expenditure per capita": "data/Current health expenditure per capita.csv",
    "Life expectancy" : "data/Life expectancy .csv",
    "Urban population": "data/Urban population.csv",
}       # the file as saved in your repository's data/ folder

SOURCE_SERIES = {
    "GDP per capita": "NY.GDP.PCAP.PP.KD",
    "Current health expenditure per capita": "SH.XPD.CHEX.PP.CD",
    "Life expectancy": "SP.DYN.LE00.IN",
    "Urban population": "SP.URB.TOTL.IN.ZS",
}
for name in FILE_NAME:
    print(name, SOURCE_SERIES[name], FILE_NAME[name], SOURCE_URL[name])

GDP per capita NY.GDP.PCAP.PP.KD data/GDP per capita.csv https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.KD
Current health expenditure per capita SH.XPD.CHEX.PP.CD data/Current health expenditure per capita.csv https://data.worldbank.org/indicator/SH.XPD.CHEX.PP.CD
Life expectancy SP.DYN.LE00.IN data/Life expectancy .csv https://data.worldbank.org/indicator/SP.DYN.LE00.IN
Urban population SP.URB.TOTL.IN.ZS data/Urban population.csv https://data.worldbank.org/indicator/SP.URB.TOTL.IN.ZS


In [55]:
aggregate_codes = [
    "AFE", "AFW", "ARB", "CEB", "CSS", "EAP", "EAR", "EAS", "ECA", "ECS",
    "EMU", "EUU", "FCV", "HIC", "HPC", "IBD", "IBT", "IDA", "IDB", "IDX",
    "INX", "LAC", "LCN", "LDC", "LIC", "LMC", "LMY", "LTE", "MEA", "MIC",
    "MNA", "NAC", "OED", "OSS", "PRE", "PSS", "PST", "SAS", "SSA", "SSF",
    "SST", "TEA", "TEC", "TLA", "TMN", "TSA", "TSS", "UMC", "WLD"
]

life_2019 = life_raw[life_raw["Country Code"].isin(aggregate_codes) == False]
life_2019 = life_2019[["Country Name", "Country Code", "2019"]]
life_2019 = life_2019.rename(columns={"2019": "life_expectancy"})

gdp_2019 = gdp_raw[gdp_raw["Country Code"].isin(aggregate_codes) == False]
gdp_2019 = gdp_2019[["Country Code", "2019"]]
gdp_2019 = gdp_2019.rename(columns={"2019": "gdp_per_capita_ppp"})

health_2019 = health_raw[health_raw["Country Code"].isin(aggregate_codes) == False]
health_2019 = health_2019[["Country Code", "2019"]]
health_2019 = health_2019.rename(columns={"2019": "health_spending_per_capita_ppp"})

urban_2019 = urban_raw[urban_raw["Country Code"].isin(aggregate_codes) == False]
urban_2019 = urban_2019[["Country Code", "2019"]]
urban_2019 = urban_2019.rename(columns={"2019": "urban_population_pct"})

print("2019 economy rows:", len(life_2019), len(gdp_2019),
      len(health_2019), len(urban_2019))
print("Missing life expectancy:", life_2019["life_expectancy"].isna().sum())
print("Missing GDP per person:", gdp_2019["gdp_per_capita_ppp"].isna().sum())
print("Missing health spending:", health_2019["health_spending_per_capita_ppp"].isna().sum())
print("Missing urban population:", urban_2019["urban_population_pct"].isna().sum())

2019 economy rows: 217 217 217 217
Missing life expectancy: 0
Missing GDP per person: 18
Missing health spending: 24
Missing urban population: 0


**Who collected it, for what purpose, what is missing and why?**

The World Bank’s World Development Indicators uses data from several sources to compare development across economies. Life expectancy comes from UN and national statistics; health spending from the WHO; GDP per capita from national accounts and purchasing power estimates; and urbanization from UN estimates.

In my 2019 files, 217 economies remain after excluding aggregates. GDP per capita is missing for 18 of them, and health spending per capita for 24. Life expectancy and urbanization have no gaps, so 186 economies have all four measures. Several missing GDP and health spending values are from small territories or economies with limited statistical reporting. The gaps may therefore be related to how much data an economy reports, though the files cannot explain each missing value. Excluding these economies could make the results less representative of them.

## Section 1: The Question and the Data (20 pts)

State the question in one sentence, and say why it matters and to whom. Describe the data as a product of measurement decisions, name the missingness mechanism (MCAR, MAR or MNAR) and defend the label. Build a summary-statistics table a reader can read without asking you a question.

In [56]:
df = life_2019.merge(gdp_2019, on="Country Code", how="left")
df = df.merge(health_2019, on="Country Code", how="left")
df = df.merge(urban_2019, on="Country Code", how="left")

df = df.rename(columns={"Country Name": "country_name",
                        "Country Code": "country_code"})
df["year"] = 2019

print("Rows and columns:", df.shape)
print("Different economies:", df["country_code"].nunique())
print("Missing values in each column:")
print(df.isna().sum())

complete = df.dropna()
print("Complete rows:", len(complete))


summary = complete[["life_expectancy", "health_spending_per_capita_ppp",
                    "urban_population_pct", "gdp_per_capita_ppp"]]
summary = summary.rename(columns={
    "life_expectancy": "Life expectancy (years)",
    "health_spending_per_capita_ppp": "Health spending per person (current PPP $)",
    "urban_population_pct": "Urban population (%)",
    "gdp_per_capita_ppp": "GDP per person (constant 2021 PPP $)"
})
print("Summary statistics for complete rows:")
display(summary.describe().round(2))

Rows and columns: (217, 7)
Different economies: 217
Missing values in each column:
country_name                       0
country_code                       0
life_expectancy                    0
gdp_per_capita_ppp                18
health_spending_per_capita_ppp    24
urban_population_pct               0
year                               0
dtype: int64
Complete rows: 186
Summary statistics for complete rows:


,Life expectancy (years),Health spending per person (current PPP $),Urban population (%),GDP per person (constant 2021 PPP $)
count,186.00,186.00,186.00,186.00
mean,72.28,1610.30,59.60,24148.58
std,8.03,1929.51,22.33,24459.40
min,31.53,36.11,13.93,1054.73
25%,66.47,227.90,41.53,6106.25
50%,73.01,821.59,61.18,15390.42
75%,77.93,2215.26,77.35,34928.62
max,85.26,10550.22,100.00,132724.75


In [57]:
# Section 1 self-check: run once your data is loaded into a DataFrame called df.
# It prints what Section 1 has to report. It doesn't judge your choices: answer the questions below.
UNIT = "country in 2019"        # one row is one what? e.g. "county" or "country-year"
OUTCOME = "life_expectancy"     # your dependent variable's column name
MAIN_X = "health_spending_per_capita_ppp"      # your main explanatory variable's column name
CONTROLS = ["urban_population_pct", "gdp_per_capita_ppp"]    # every other column in your model

if "df" in globals():
    print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   One row is: {UNIT or '(fill in UNIT)'}")
    cols = [c for c in [OUTCOME, MAIN_X, *CONTROLS] if c]
    absent = [c for c in cols if c not in df.columns]
    if absent:
        print("Not in df:", absent)
    present = [c for c in cols if c in df.columns]
    if present:
        print("\nMissing values per column:")
        print(df[present].isna().sum().to_string())
    if OUTCOME in df.columns and pd.api.types.is_numeric_dtype(df[OUTCOME]):
        print(f"\nSkew of {OUTCOME}: {df[OUTCOME].skew():.2f} (above about 1, consider a log)")
else:
    print("Load your data into a DataFrame called df first, then run this cell.")

Rows: 217   Columns: 7   One row is: country in 2019

Missing values per column:
life_expectancy                    0
health_spending_per_capita_ppp    24
urban_population_pct               0
gdp_per_capita_ppp                18

Skew of life_expectancy: -0.95 (above about 1, consider a log)


**Answer before you go on.**

1. For each control: does your main explanatory variable change it? If yes, it is on the path from X to Y. Drop it, or run with and without and say what each version measures.
2. Are all your variables in the same units and for the same years?
3. If your question is about a policy, where is the period before it?

*Your answers here.*

1. I include GDP per capita because wealthier economies may spend more on health and tend to have longer life expectancy. Income may also be related to urbanization. Since health spending and urbanization could affect income over time, controlling for GDP might remove part of their longer-run relationship with life expectancy. I will therefore describe the results as associations, not causal effects.

2. Each row represents a country or economy in 2019. Health spending is measured in current PPP international dollars per person, while GDP per capita is measured in constant 2021 PPP international dollars per person. Both adjust for price differences across countries, but they use different price bases. I will log them separately rather than divide one by the other.

3. I chose 2019 for a cross-country comparison before the COVID-19 pandemic affected life expectancy. This question does not involve a policy change or a before-and-after comparison.

*Your words here.*

Researchers and policymakers may want to know whether economies that spend more on health, or have a larger urban population, also have longer life expectancy. This comparison can show how those measures vary together. It cannot tell us what would happen if a government increased health spending.

I use four files from the World Bank’s World Development Indicators. The World Bank brings together figures from several sources: UN and national statistics for life expectancy, the WHO for health spending, national accounts and purchasing power estimates for GDP per person, and UN estimates for urban population share. How each source defines and reports these measures affects the numbers, especially because countries use their own definitions of urban areas.

I kept 2019, removed regional and income group totals, and joined the files by country code. The result has 217 rows and 7 columns, with one row per economy. GDP per person is missing for 18 economies and health spending for 24. Life expectancy and urbanization have no missing values. Some gaps overlap, leaving 186 economies with all four measures. Several missing values belong to small territories, and some economies missing GDP data are affected by conflict. I would not assume the gaps are missing completely at random (MCAR). Missing at random (MAR) is possible if observed features of an economy explain the gaps, but the files cannot rule out missing not at random (MNAR). Leaving those economies out may make the results less representative of them.

The table above describes the 186 complete cases. It gives the number of observations, mean, standard deviation, median, and range for each measure, with units in the row labels. The lowest life expectancy is 31.53 years for the Central African Republic. That value is unusually low, so I will check how much it affects the model in the diagnostics.

## Section 2: Descriptive Evidence (20 pts)

Show the reader the data before any model. At least two figures, each with a caption that states its finding, not its variables. Chapter 3's standards for axes. If the outcome is skewed, show it and say what you will do about it.

*Your words here.*

## Section 3: The Model (25 pts)

Fit a multivariate OLS model and report it in a formatted regression table (coefficients, standard errors, N, R-squared), not a `summary()` dump. Interpret at least two coefficients in full: magnitude, units, direction, precision. For every control, say whether it blocks a confounding path or adds precision, and what would make it a bad control (Chapter 10). OLS only: no difference-in-differences estimate.

*Your words here.*

## Section 4: Diagnostics (20 pts)

Residuals against fitted values and against your main predictor; a constant-variance check, with robust standard errors if it fails; variance inflation factors, interpreted; at least one influential-observation check.

**Start only after Topic 14.** Write Sections 1 to 3 first; they do not depend on it.

*Your words here.*

## Section 5: Threats and Conclusion (10 pts)

First the threats: what would have to be true for your main coefficient to be causal, is it, and if not, what your estimate is instead. Then the conclusion: what you found, how confident a reader should be, and what you would do with another month. Half a page. No new results, so there is no code cell here.

*Your words here.*

## Appendix

The regression table, the diagnostics output and any extra figures. The appendix does not count toward the 6–8 page limit of the report.

## Numbers my report quotes

Every number in your report, each with a name that says where it comes from. A number in the report that this cell does not print is unsupported, and is graded as if it were missing. Keep it the last code cell.

In [54]:
REPORT_NUMBERS = {
    # "Section 1: rows in the final sample": ...,
    # "Section 3: coefficient on <your main predictor>": ...,
}

for name, value in REPORT_NUMBERS.items():
    print(f"{name}: {value}")

## Checkpoints

In Colab, *File › Save a copy in GitHub* makes a commit each time you use it. Save at each checkpoint so the commit history shows it.

- [ ] **Mon 26 Oct:** your question and data source, posted in the Midterm checkpoint 1 discussion.
- [ ] **Mon 2 Nov:** your data loaded and your summary table, committed to your repository.
- [ ] **Fri 6 Nov:** a draft of your model, committed.
- [ ] **Mon 9 Nov, 11:59pm:** the project is due.

## Before you submit

- [ ] A Colab notebook with all your analysis, cleanly organised.
- [ ] A written report, 6 to 8 pages of text, uploaded as a PDF.
- [ ] A GitHub repository containing both, with a `README.md` you wrote yourself.
- [ ] **Question and data provenance (20):** A defensible measurable question; data described as a measurement product; missingness mechanism identified and defended; interpretable summary table
- [ ] **Descriptive evidence (20):** Two or more figures each making a stated point; captions state findings not variables; Chapter 3 standards met
- [ ] **Model and interpretation (25):** Properly formatted regression table; two coefficients interpreted in full including units and magnitude; every control justified by role
- [ ] **Diagnostics (20):** Residuals, heteroskedasticity, VIF and influence all run and interpreted; robust errors used where warranted
- [ ] **Threats and conclusion (10):** The causal claim assessed in plain terms; the estimand correctly labelled if not causal; conclusion adds no new results
- [ ] **Repository, checkpoints and reproducibility (5):** Public GitHub repo containing the notebook and the report; the two committed checkpoints visible in the commit history; data included or its download documented; a self-written README giving the question, the source, how to run it and the headline finding
- [ ] The Colab notebook is shared with *Anyone with the link*, and the link is in the Canvas submission box.
- [ ] Every number in the report is printed by the *Numbers my report quotes* cell.
- [ ] No AI anywhere: code, analysis, writing, editing, README.
- [ ] *File › Save a copy in GitHub*, path `Midterm Project/Econ_3916_Midterm.ipynb`.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.